In [ ]:
import pandas as pd
import geopandas as gpd

import sys
sys.path.append('..')
from helpers import (
    get_bulk_power_sales,
    get_county2zone,
    get_service_territories
)

In [ ]:
bulk_power_sales_by_year = {}
county2zone_by_year = {}
service_territories_by_year = {}

for load_year in range(2016, 2024):
    bulk_power_sales_by_year[load_year] = get_bulk_power_sales(load_year, {})
    county2zone = get_county2zone(load_year)
    service_territories = get_service_territories(
        county2zone, load_year
    )

    county2zone_by_year[load_year] = county2zone
    service_territories_by_year[load_year] = service_territories.merge(
        county2zone[['county_state', 'FIPS']],
        on='county_state',
        how='left'
    )

def calculate_county_load_estimates(load_year):
    bulk_power_sales = bulk_power_sales_by_year[load_year].copy()
    county2zone = county2zone_by_year[load_year].copy()
    service_territories = service_territories_by_year[load_year].copy()

    county2zone = county2zone.merge(
        (
            bulk_power_sales.groupby('State', as_index=False)
            ['Megawatthours']
            .sum(numeric_only=True)
            .rename(columns={'State': 'state', 'Megawatthours': 'state_sales_mwh'})
        ),
        on='state'
    )
    county2zone['estimated_sales_mwh'] = (
        county2zone['state_sales_mwh']
        * county2zone['population']
        / county2zone.groupby('state')['population'].transform('sum')
    )

    county_load_estimates = county2zone.set_index('FIPS')['estimated_sales_mwh']

    return county_load_estimates

county_load_estimates_by_year = {}
for load_year in range(2016, 2024):
    county_load_estimates_by_year[load_year] = (
        calculate_county_load_estimates(load_year)
    )

county_load_estimates = pd.concat(county_load_estimates_by_year, axis=1)

county_load_estimates.to_csv('../data/county_ftm_sales_estimates.csv')